# Model Registry with Snowflake ML

This notebook demonstrates the **Snowflake Model Registry**:

- **Log models** with dependencies, signatures, and metadata
- **Version management**: multiple versions, default version, tags
- **Batch inference**: run predictions directly from the registry
- **Explainability**: built-in SHAP feature importance
- **Lineage**: track which features fed each model

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col
from snowflake.ml.registry import Registry
from snowflake.ml.model.model_signature import infer_signature

session = get_active_session()
session.query_tag = {"origin": "ml_deep_dive", "name": "banking_fraud_detection", "notebook": "03_model_registry"}

DB = "BANKING_ML_DEMO"
SCHEMA = "FRAUD_DETECTION"
WH = "ML_DEMO_WH"

session.use_database(DB)
session.use_schema(SCHEMA)
session.use_warehouse(WH)

## Prepare Data and Train Models

Self-contained data prep and model training for this notebook.

In [ ]:
# Load and prepare features (same as notebook 02)
txn_df = session.table("TXN_FEATURES")
cust_features = session.sql("""
    SELECT CUSTOMER_ID, AVG(AMOUNT) AS CUST_AVG_TRANSACTION_AMT,
           COUNT(*) AS CUST_TOTAL_TRANSACTIONS, STDDEV(AMOUNT) AS CUST_STDDEV_AMOUNT,
           MAX(AMOUNT) AS CUST_MAX_AMOUNT, COUNT(DISTINCT MERCHANT_ID) AS CUST_UNIQUE_MERCHANTS,
           AVG(IS_ONLINE) AS CUST_PCT_ONLINE, AVG(CUSTOMER_AGE) AS CUST_AGE,
           AVG(CUSTOMER_INCOME) AS CUST_INCOME
    FROM RAW_TRANSACTIONS GROUP BY CUSTOMER_ID
""")
merch_features = session.sql("""
    SELECT MERCHANT_ID, AVG(AMOUNT) AS MERCH_AVG_TRANSACTION_AMT,
           COUNT(*) AS MERCH_TOTAL_TRANSACTIONS, AVG(IS_FRAUD) AS MERCH_FRAUD_RATE
    FROM RAW_TRANSACTIONS GROUP BY MERCHANT_ID
""")

pdf = txn_df.join(cust_features, "CUSTOMER_ID").join(merch_features, "MERCHANT_ID").to_pandas()

FEATURE_COLS = [
    "AMOUNT", "IS_ONLINE", "TXN_HOUR", "TXN_DAY_OF_WEEK", "TXN_IS_WEEKEND",
    "TXN_IS_LATE_NIGHT", "TXN_AMOUNT_RATIO", "TXN_IS_HIGH_AMOUNT",
    "CUST_AVG_TRANSACTION_AMT", "CUST_TOTAL_TRANSACTIONS", "CUST_STDDEV_AMOUNT",
    "CUST_MAX_AMOUNT", "CUST_UNIQUE_MERCHANTS", "CUST_PCT_ONLINE",
    "CUST_AGE", "CUST_INCOME",
    "MERCH_AVG_TRANSACTION_AMT", "MERCH_TOTAL_TRANSACTIONS", "MERCH_FRAUD_RATE"
]

X = pdf[FEATURE_COLS].fillna(0).astype(float)
y = pdf["IS_FRAUD"].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
scale_pos = len(y_train[y_train == 0]) / max(len(y_train[y_train == 1]), 1)

print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")

In [ ]:
# Train baseline model
xgb_baseline = XGBClassifier(
    n_estimators=100, max_depth=4, learning_rate=0.1,
    scale_pos_weight=scale_pos, eval_metric="aucpr", random_state=42
)
xgb_baseline.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

y_pred_bl = xgb_baseline.predict(X_test)
y_proba_bl = xgb_baseline.predict_proba(X_test)[:, 1]
bl_metrics = {
    "auc": roc_auc_score(y_test, y_proba_bl),
    "f1": f1_score(y_test, y_pred_bl),
    "precision": precision_score(y_test, y_pred_bl),
    "recall": recall_score(y_test, y_pred_bl)
}

# Train tuned model
xgb_tuned = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    min_child_weight=5, subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=scale_pos, eval_metric="aucpr", random_state=42
)
xgb_tuned.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

y_pred_tu = xgb_tuned.predict(X_test)
y_proba_tu = xgb_tuned.predict_proba(X_test)[:, 1]
tu_metrics = {
    "auc": roc_auc_score(y_test, y_proba_tu),
    "f1": f1_score(y_test, y_pred_tu),
    "precision": precision_score(y_test, y_pred_tu),
    "recall": recall_score(y_test, y_pred_tu)
}

print(f"Baseline - AUC: {bl_metrics['auc']:.4f}, F1: {bl_metrics['f1']:.4f}")
print(f"Tuned    - AUC: {tu_metrics['auc']:.4f}, F1: {tu_metrics['f1']:.4f}")

## Initialize Model Registry

In [ ]:
reg = Registry(
    session=session,
    database_name=DB,
    schema_name=SCHEMA
)

print("Model Registry initialized")
print(f"Existing models: {reg.show_models()}")

## Log Model Version 1 (Baseline)

In [ ]:
mv_baseline = reg.log_model(
    model=xgb_baseline,
    model_name="FRAUD_DETECTION_MODEL",
    version_name="V1_BASELINE",
    conda_dependencies=["scikit-learn", "xgboost"],
    sample_input_data=X_test.head(10),
    comment="Baseline XGBoost fraud detection model. Default hyperparameters, 100 estimators, max_depth=4.",
    metrics={k: float(v) for k, v in bl_metrics.items()}
)

print("Logged FRAUD_DETECTION_MODEL version V1_BASELINE")
print(f"Metrics: {bl_metrics}")

## Log Model Version 2 (Tuned)

In [ ]:
mv_tuned = reg.log_model(
    model=xgb_tuned,
    model_name="FRAUD_DETECTION_MODEL",
    version_name="V2_TUNED",
    conda_dependencies=["scikit-learn", "xgboost"],
    sample_input_data=X_test.head(10),
    comment="Tuned XGBoost: 300 estimators, max_depth=6, lr=0.05, subsample=0.8, regularized.",
    metrics={k: float(v) for k, v in tu_metrics.items()}
)

print("Logged FRAUD_DETECTION_MODEL version V2_TUNED")
print(f"Metrics: {tu_metrics}")

## Explore Model Registry

In [ ]:
# List all models
print("=== Registered Models ===")
print(reg.show_models())

# Get model and list versions
model = reg.get_model("FRAUD_DETECTION_MODEL")
print("\n=== Model Versions ===")
print(model.show_versions())

# Version details
mv = model.version("V2_TUNED")
print(f"\n=== V2_TUNED Details ===")
print(f"Comment: {mv.comment}")
print(f"Metrics: {mv.show_metrics()}")

## Batch Inference with Model Registry

Run predictions directly from the registry - no need to load the model locally.

In [ ]:
mv = reg.get_model("FRAUD_DETECTION_MODEL").version("V2_TUNED")

# Batch predict on test data
sample = X_test.head(20).reset_index(drop=True)
predictions = mv.run(sample, function_name="predict")
probabilities = mv.run(sample, function_name="predict_proba")

result_df = sample.copy()
result_df["PREDICTION"] = predictions["output_feature_0"].values
result_df["ACTUAL"] = y_test.head(20).values

print("=== Batch Inference Results ===")
print(result_df[["AMOUNT", "TXN_HOUR", "TXN_IS_LATE_NIGHT", "TXN_AMOUNT_RATIO", "PREDICTION", "ACTUAL"]].to_string())

## Model Explainability (SHAP)

Snowflake Model Registry provides built-in explainability via SHAP values.

In [ ]:
# Compute SHAP values using the registry's explain function
sample_data = X_test.sample(n=500, random_state=42).reset_index(drop=True)

try:
    shap_values = mv.run(sample_data, function_name="explain")
    print(f"SHAP values shape: {shap_values.shape}")
    print(shap_values.head())
except Exception as e:
    print(f"Built-in explain not available, using local SHAP: {e}")
    import shap
    explainer = shap.TreeExplainer(xgb_tuned)
    shap_values = explainer.shap_values(sample_data)
    shap_df = pd.DataFrame(shap_values, columns=FEATURE_COLS)
    print(f"SHAP values shape: {shap_df.shape}")

In [ ]:
# Feature importance visualization
import shap
explainer = shap.TreeExplainer(xgb_tuned)
shap_vals = explainer.shap_values(sample_data)

mean_abs_shap = pd.Series(
    np.abs(shap_vals).mean(axis=0),
    index=FEATURE_COLS
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
mean_abs_shap.plot(kind="barh", ax=ax, color="#29B5E8")
ax.set_xlabel("Mean |SHAP Value|")
ax.set_title("Feature Importance - Fraud Detection Model (V2_TUNED)")
plt.tight_layout()
plt.show()

print("\nTop 5 most important features:")
for feat, val in mean_abs_shap.tail(5).items():
    print(f"  {feat}: {val:.4f}")

## Set Default Model Version

In [ ]:
model = reg.get_model("FRAUD_DETECTION_MODEL")
model.default = model.version("V2_TUNED")

print(f"Default version set to: V2_TUNED")
print(f"Verified default: {model.default.version_name}")

## View in Snowsight

Navigate to **Snowsight > AI & ML > Models** to explore:
- Registered models and their versions
- Version comparison with metrics
- Model lineage (which Feature Views fed the model)
- Built-in explainability visualizations

### Next Steps

Proceed to **`04_model_serving.ipynb`** to deploy the model as a real-time inference service.